# Aula01 — LangChain

Uma seção por vídeo, cada uma autossuficiente. Requer o arquivo `.env` (veja `.example.env`).
Os `main.py` originais continuam em `video0X/`.

## video01

In [ ]:
import os

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# CONFIGURAÇÃO DE AMBIENTE
# Carrega as variáveis de ambiente do arquivo .env
load_dotenv()

# Validação para garantir que a chave de API foi carregada
if not (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")):
    raise ValueError("A variável de ambiente GOOGLE_API_KEY não foi encontrada. Crie um arquivo .env e adicione-a.")

print("--- Inicializando o Modelo e a Prompt Template ---")

# 2. Definindo o modelo de linguagem (LLM)
model = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente técnico especialista em arquitetura de software de Inteligência Artificial. Seja direto e profissional."),
    ("user", "Explique brevemente o conceito de {topico} e por que ele é modular.")
])

output_parser = StrOutputParser()

chain = prompt_template | model | output_parser

topico_exemplo = "LangChain Expression Language (LCEL)"
print(f"\n[Executando Chain] Solicitando explicação sobre: {topico_exemplo}...")  

resposta = chain.invoke({"topico": topico_exemplo})

print("\n=== RESPOSTA DA CHAIN INTERNA ===")
print(resposta)

## video02

In [ ]:
import logging
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
# [notebook] no LangChain 1.x o AgentExecutor foi movido para o pacote langchain-classic
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage

# Silencia avisos de schema não suportado para limpar o console
logging.getLogger("langchain_google_genai").setLevel(logging.ERROR)

load_dotenv()

print("--- Configurando um Agente com Memória e Ferramentas Customizadas ---")

@tool
def calcular_custo_tokens(quantidade_tokens: int, modelo: str) -> str:
    """Calcula o custo estimado em dólares com base na quantidade de tokens e no modelo especificado."""
    modelo = modelo.lower()
    if "gpt-4o-mini" in modelo:
        custo = (quantidade_tokens / 1_000_000) * 0.15 # Exemplo de preço hipotético por milhão
        return f"O custo estimado para {quantidade_tokens} tokens no gpt-4o-mini é de ${custo:.6f} USD."
    else:
        return "Modelo não mapeado para cálculo de custo local."

# Lista de ferramentas disponíveis para o agente
tools = [calcular_custo_tokens]

llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)

# Criando o Prompt do Agente contendo um espaço reservado para o histórico (Memória)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um agente financeiro de IA. Use as ferramentas disponíveis para responder dúvidas sobre custos se necessário."),
    MessagesPlaceholder(variable_name="chat_history"), # Onde o histórico da conversa reside
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"), # Onde o agente "pensa" e planeja as ações
])

agent = create_tool_calling_agent(llm, tools, prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

historico_conversa = [
    HumanMessage(content="Olá, meu nome é Carlos e estou construindo um assistente."),
    AIMessage(content="Olá Carlos! Como posso te ajudar com os custos ou arquitetura do seu assistente hoje?")
]

pergunta_usuario = "Se meu sistema rodar 5.000.000 de tokens no gpt-4o-mini, quanto vou gastar?"

print(f"\n[Usuário]: {pergunta_usuario}")
resposta_final = agent_executor.invoke({
    "input": pergunta_usuario,
    "chat_history": historico_conversa
})

print("\n=== RESPOSTA FINAL DO AGENTE ===")
print(resposta_final["output"])

## video03

In [ ]:
# Script_Video_03.py
import os
import logging
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilyAnswer
# [notebook] no LangChain 1.x o AgentExecutor foi movido para o pacote langchain-classic
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

logging.getLogger("langchain_google_genai").setLevel(logging.ERROR)

load_dotenv()

# Validação para garantir que a chave de API foi carregada
if not (os.getenv("GOOGLE_API_KEY") and os.getenv("TAVILY_API_KEY")):
    raise ValueError("A variável de ambiente GOOGLE_API_KEY e TAVILY_API_KEY não foram encontradas.\
                      Crie um arquivo .env e adicione-a.")

print("--- Iniciando o Projeto Prático: Answer Engine ---")

# 1. Instanciando uma ferramenta de busca em tempo real na internet
busca_internet = TavilyAnswer()
ferramentas_projeto = [busca_internet]

# 2. Inicializando a LLM que atuará como controladora
llm_controladora = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash", 
    temperature=0
)

# 3. Definindo o prompt de comportamento do Answer Engine
prompt_engine = ChatPromptTemplate.from_messages([
    ("system", (
        "Você é o 'Answer Engine', um motor de respostas avançado integrado à internet. "
        "Sempre que o usuário perguntar sobre fatos recentes, ações de mercado, notícias ou cotações, "
        "use a ferramenta de busca para trazer dados verídicos e cite as descobertas de forma resumida."
    )),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# 4. Montando o agente focado em Tool Binding (vincular ferramentas ao modelo)
engine_agent = create_tool_calling_agent(llm_controladora, ferramentas_projeto, prompt_engine)

# 5. Criando o executor do Answer Engine
answer_engine_executor = AgentExecutor(
    agent=engine_agent, 
    tools=ferramentas_projeto, 
    verbose=True, # Exibe no terminal os passos de "Thought -> Action -> Observation"
    handle_parsing_errors=True # Garante resiliência caso o formato de saída quebre
)

# 6. Execução do teste prático (Uma pergunta factual recente que a LLM não saberia sem a ferramenta)
pergunta_atual = "Quem ganhou o último campeonato de Fórmula 1 e qual foi a pontuação final?"

print(f"\n[Consultando Answer Engine]: {pergunta_atual}")
resultado_engine = answer_engine_executor.invoke({"input": pergunta_atual})

print("\n=== RESPOSTA ENRICHED (COM BUSCA EM TEMPO REAL) ===")
print(resultado_engine["output"])